# Lezione 2 — Dati e dataset in PyTorch

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife (Hub di Innovazione Digitale Europeo), basato sulla certificazione EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Durata stimata della lezione:** 100 minuti.

## Obiettivi di apprendimento di questa lezione

- Spiegare perché i dati sono la base dell'addestramento e quali formati supporta PyTorch
- Usare la classe `Dataset` per rappresentare un insieme di dati
- Usare `DataLoader` per servire i dati in batch durante l'addestramento
- Caricare un dataset di riferimento (MNIST) con `torchvision` e applicare trasformazioni di base

Questo notebook accompagna il capitolo 02 del manuale del corso. Esegui ogni cella in ordine con **Maiusc + Invio**. Puoi eseguirlo così com'è in Google Colab: ha bisogno solo di `torch`, `torchvision` e `matplotlib`, e la prima volta che carichi MNIST verrà scaricato automaticamente.

## Parte 1 — Perché questo argomento è importante

Nella lezione precedente hai costruito una rete neurale minima con `nn.Module`. Quella rete non impara nulla da sola: ha bisogno di dati con cui allenarsi. Nel Deep Learning (DL, apprendimento profondo), preparare, caricare e iterare sui dati generalmente consuma più tempo della progettazione dell'architettura della rete — si stima che possa rappresentare il 90% del lavoro reale di un progetto di Intelligenza Artificiale (IA).

Prima di addestrare qualcosa, PyTorch ha bisogno di rispondere a tre domande sui tuoi dati:

- In quale formato sono e come sono rappresentati come tensor?
- Come si dividono in un insieme di addestramento e uno di test?
- Come si consegnano al modello durante l'addestramento senza esaurire la memoria del computer?

PyTorch risolve questo con due componenti che lavorano insieme: `Dataset` e `DataLoader`.

## Parte 2 — La classe Dataset

Un `Dataset` è un contratto molto semplice: qualsiasi oggetto che lo rispetta sa come rispondere a due domande:

- `__len__`: quanti elementi ha in totale?
- `__getitem__`: com'è l'elemento numero *i*?

Non importa se sotto il cofano c'è una lista di numeri o diecimila immagini su disco: finché l'oggetto risponde a quelle due domande, PyTorch lo tratta come un dataset. Puoi costruire il tuo `Dataset` ereditando da `torch.utils.data.Dataset`, oppure usare uno di quelli già pronti in `torchvision`. Oggi cominci con uno già fatto — MNIST — per concentrarti sul meccanismo prima di costruire i tuoi dati.

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print("Versión de torch:", torch.__version__)
print("Versión de torchvision:", torchvision.__version__)

## Parte 3 — Cosa offre torchvision

`torchvision` è il modulo di PyTorch focalizzato sulla visione artificiale. Offre due cose che userai proprio adesso:

- **Dataset di riferimento** già confezionati come classi `Dataset`: MNIST (cifre scritte a mano), CIFAR-10 (immagini a colori di dieci categorie) e ImageNet (milioni di immagini con migliaia di categorie), tra gli altri
- **Trasformazioni**, in `torchvision.transforms`, che convertono i dati grezzi nel formato che il modello si aspetta. Qui usi quella più elementare: `transforms.ToTensor()`, che converte un'immagine in un tensor di PyTorch

Caricherai MNIST: la parte di addestramento e quella di test, separatamente. Se è la prima volta che esegui questa cella, `download=True` scaricherà i dati nella cartella `./dati`.

In [ ]:
# Transformación básica: convierte cada imagen (en formato PIL) en un tensor de PyTorch
transformacion = transforms.ToTensor()

# Dataset de entrenamiento: se usa para ajustar los parámetros del modelo
trainset = datasets.MNIST(
    root="./datos",
    train=True,
    download=True,
    transform=transformacion,
)

# Dataset de prueba (test): datos que el modelo nunca ve durante el entrenamiento,
# reservados para comprobar si generaliza bien
testset = datasets.MNIST(
    root="./datos",
    train=False,
    download=True,
    transform=transformacion,
)

print("Ejemplos de entrenamiento:", len(trainset))
print("Ejemplos de test:", len(testset))

## Parte 4 — DataLoader e addestramento per batch

Avere il dataset organizzato non basta: durante l'addestramento devi servirlo al modello in porzioni gestibili, chiamate **batch**. `torch.utils.data.DataLoader` raggruppa il dataset in batch di una dimensione che tu decidi (`batch_size`) e ti consente di iterare su di essi con un semplice ciclo `for`.

Addestrare per batch ha parecchi vantaggi rispetto all'uso dell'intero dataset in una sola volta:

- Un dataset grande potrebbe non stare completamente in memoria; un batch sì
- Il modello aggiorna i suoi parametri varie volte per ogni passata completa (*epoch*) attraverso il dataset, invece che una sola volta
- Una dimensione di batch tipica è tra otto e 64 esempi

Il parametro `shuffle=True` mescola l'ordine degli esempi prima di ogni epoch. Questo è particolarmente importante in MNIST: se il modello vedesse prima tutti gli "zero", poi tutti gli "uno", e così via, potrebbe imparare scorciatoie legate a quella posizione invece che alle caratteristiche reali di ogni cifra.

In [ ]:
# DataLoader de entrenamiento: lotes de 10 ejemplos, mezclados en cada época
trainloader = DataLoader(trainset, batch_size=10, shuffle=True)

# DataLoader de test: mismo tamaño de lote, también mezclado (aquí el orden es menos crítico)
testloader = DataLoader(testset, batch_size=10, shuffle=True)

print("Número de lotes de entrenamiento:", len(trainloader))
print("Número de lotes de test:", len(testloader))

In [ ]:
# Iteramos un único lote para inspeccionarlo
for datos in trainloader:
    X, y = datos  # X: imágenes del lote, y: etiquetas (el dígito de cada imagen)
    break  # con el primer lote nos basta, no queremos recorrer todo el dataset

print("Forma del lote de imágenes:", X.shape)   # [batch_size, canales, alto, ancho]
print("Forma del lote de etiquetas:", y.shape)
print("Etiquetas del lote:", y)

## Parte 5 — Visualizzare un batch

Ogni immagine del batch ha forma `(1, 28, 28)`: un canale di colore (scala di grigi) e 28x28 pixel. Per disegnarla con `matplotlib` devi rimuovere la dimensione del canale con `.view(28, 28)` o `.squeeze()`. Mostriamo le dieci immagini del batch insieme alla loro etichetta.

In [ ]:
import matplotlib.pyplot as plt

figura, ejes = plt.subplots(1, 10, figsize=(15, 2))

for i in range(10):
    imagen = X[i][0].view(28, 28)  # quitamos la dimensión del canal: (1, 28, 28) -> (28, 28)
    ejes[i].imshow(imagen, cmap="gray")
    ejes[i].set_title(str(y[i].item()))
    ejes[i].axis("off")

plt.tight_layout()
plt.show()

## Parte 6 — È equilibrato il dataset?

Un dataset è **equilibrato** quando ogni categoria (qui, ogni cifra da zero a nove) ha un numero simile di esempi. Se una categoria domina il dataset, il modello potrebbe imparare la scorciatoia di prevedere sempre quella opzione per minimizzare l'errore, invece di imparare a distinguere davvero tra cifre.

Contiamo quanti esempi ci sono di ogni cifra nell'insieme di addestramento, iterando il dataset con il `DataLoader`.

In [ ]:
from collections import Counter

contador = Counter()

for _, etiquetas in trainloader:
    contador.update(etiquetas.tolist())

total = sum(contador.values())
for digito in sorted(contador):
    porcentaje = 100 * contador[digito] / total
    print(f"Dígito {digito}: {contador[digito]} ejemplos ({porcentaje:.1f}%)")

## Riepilogo della lezione

- I dati sono la base dell'addestramento: prepararli e organizzarli generalmente richiede più tempo che progettare la rete
- Un `Dataset` è qualsiasi oggetto che sa quanti elementi ha (`__len__`) e come restituire l'esempio numero *i* (`__getitem__`)
- `torchvision` offre dataset di riferimento (MNIST, CIFAR-10, ImageNet) e trasformazioni (`torchvision.transforms`) pronte all'uso
- `DataLoader` serve i dati in batch (`batch_size`), consente di mescolarli (`shuffle=True`) ed evita di caricare l'intero dataset in memoria in una volta
- Verificare se un dataset è equilibrato — contando esempi per classe — aiuta a rilevare bias prima dell'addestramento

Nel notebook di pratica costruirai il tuo `Dataset` personalizzato e calcolerai quanti batch risultano dividendo MNIST con diversi valori di `batch_size`.